# Section 7: Recommender Systems from Implicit Feedback

**What you'll learn**
- Turn MovieLens ratings into implicit feedback with no labels
- Hold out each user's last movies and grade with recall@10 and NDCG@10
- Train popularity, item similarity, ALS, EASE and SASRec with one library, RecTools
- Find "more like this" movies
- Ask SASRec what comes next after your own watch history

## Setup
- Slides 1-2
- RecTools gives every model the same `fit` and `recommend`
- Item similarity and ALS come from the `implicit` library, wrapped by RecTools
- The last three lines hide library warnings and PyTorch Lightning's start-up messages, and let the GPU use its fast matrix maths

In [1]:
import logging
import time
import urllib.request
import warnings
import zipfile
import pandas as pd
import torch
from threadpoolctl import threadpool_limits
from implicit.nearest_neighbours import CosineRecommender
from implicit.als import AlternatingLeastSquares
from rectools.dataset import Dataset
from rectools.metrics import Recall, NDCG, calc_metrics
from rectools.models import PopularModel, ImplicitItemKNNWrapperModel, ImplicitALSWrapperModel, EASEModel, SASRecModel

warnings.filterwarnings("ignore")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
torch.set_float32_matmul_precision("high")

## Loading MovieLens 1M
- One million ratings from 6,040 users on about 3,700 movies, with timestamps

In [2]:
urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-1m.zip", "ml-1m.zip")
z = zipfile.ZipFile("ml-1m.zip")
z.namelist()

['ml-1m/',
 'ml-1m/movies.dat',
 'ml-1m/ratings.dat',
 'ml-1m/README',
 'ml-1m/users.dat']

- RecTools reads four columns: `user_id`, `item_id`, `weight` and `datetime`

In [3]:
ratings = pd.read_csv(z.open("ml-1m/ratings.dat"), sep="::", engine="python",
                      names=["user_id", "item_id", "weight", "datetime"])
ratings.head()

,user_id,item_id,weight,datetime
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [4]:
movies = pd.read_csv(z.open("ml-1m/movies.dat"), sep="::", engine="python",
                     names=["item_id", "title", "genres"], encoding="latin-1")
titles = movies.set_index("item_id")["title"]
titles.head()

item_id
1                      Toy Story (1995)
2                        Jumanji (1995)
3               Grumpier Old Men (1995)
4              Waiting to Exhale (1995)
5    Father of the Bride Part II (1995)
Name: title, dtype: object

## Ratings as Implicit Feedback
- Slide 3
- Keep only who watched what: every watch counts as 1, the star rating is dropped
- Seconds since 1970 become real dates, so each user's movies can be put in order

In [5]:
ratings["weight"] = 1.0
ratings["datetime"] = pd.to_datetime(ratings["datetime"], unit="s")
ratings.head()

,user_id,item_id,weight,datetime
0,1,1193,1.0,2000-12-31 22:12:40
1,1,661,1.0,2000-12-31 22:35:09
2,1,914,1.0,2000-12-31 22:32:48
3,1,3408,1.0,2000-12-31 22:04:35
4,1,2355,1.0,2001-01-06 23:38:11


- Every user has at least 20 movies

In [6]:
ratings.groupby("user_id").size().describe()

count    6040.000000
mean      165.597517
std       192.747029
min        20.000000
25%        44.000000
50%        96.000000
75%       208.000000
max      2314.000000
dtype: float64

## Holding Out Each User's Last 10 Movies
- Slide 4
- Train on the past, predict the future, like a real system

In [7]:
ratings = ratings.sort_values(["user_id", "datetime"], kind="stable")
from_end = ratings.groupby("user_id").cumcount(ascending=False)
train = ratings[from_end >= 10]
test = ratings[from_end < 10]
len(train), len(test)

(939809, 60400)

## The Watch Matrix
- Slide 5
- `Dataset` numbers the users and movies and builds the sparse watch matrix

In [8]:
dataset = Dataset.construct(train)
X = dataset.get_user_item_matrix()
X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 939809 stored elements and shape (6040, 3700)>

- Share of the matrix that is filled

In [9]:
X.nnz / (X.shape[0] * X.shape[1])

0.04205338285305173

## Grading a Recommender
- Slide 6
- One function, used for every model: fit on the past, take each user's top 10 with movies already watched hidden
- Recall@10: share of the 10 held-out movies in the top 10
- NDCG@10: the same, with more credit near the top of the list

In [10]:
metrics = {"recall@10": Recall(k=10), "ndcg@10": NDCG(k=10)}
users = test["user_id"].unique()
results = {}

def grade(name, model):
    start = time.time()
    model.fit(dataset)
    reco = model.recommend(users=users, dataset=dataset, k=10, filter_viewed=True)
    results[name] = calc_metrics(metrics, reco=reco, interactions=test, prev_interactions=train)
    results[name]["seconds"] = round(time.time() - start, 1)
    return results[name]

## Popularity Baseline
- Slide 7
- Everyone gets the most watched movies they have not seen

In [11]:
grade("Popularity", PopularModel())

{'recall@10': 0.04104304635761589,
 'ndcg@10': 0.04443594303089803,
 'seconds': 0.2}

In [12]:
titles[train["item_id"].value_counts().index[:10]]

item_id
2858                               American Beauty (1999)
1196    Star Wars: Episode V - The Empire Strikes Back...
260             Star Wars: Episode IV - A New Hope (1977)
1210    Star Wars: Episode VI - Return of the Jedi (1983)
589                     Terminator 2: Judgment Day (1991)
480                                  Jurassic Park (1993)
2028                           Saving Private Ryan (1998)
1270                            Back to the Future (1985)
2571                                   Matrix, The (1999)
593                      Silence of the Lambs, The (1991)
Name: title, dtype: object

## Item Similarity
- Slide 8
- Two movies are similar when the same people watched them
- Each movie keeps its 20 most similar movies; a user's score adds up the similarities to everything they watched

In [13]:
grade("Item similarity", ImplicitItemKNNWrapperModel(CosineRecommender()))

{'recall@10': 0.06698675496688743,
 'ndcg@10': 0.07079467662849052,
 'seconds': 0.5}

## Implicit ALS
- Slides 9-10
- Factor the watch matrix into user vectors and item vectors
- Score = dot product of a user vector and an item vector
- implicit asks for one BLAS thread; it runs its own threads

In [14]:
threadpool_limits(1, "blas")
als = ImplicitALSWrapperModel(AlternatingLeastSquares(factors=64))
grade("ALS", als)

{'recall@10': 0.07317880794701986,
 'ndcg@10': 0.07563332950657461,
 'seconds': 0.6}

## Recommendations for One User
- The last 10 movies user 1 watched in the training data

In [15]:
user = 1
titles[train[train["user_id"] == user]["item_id"].tail(10)]

item_id
3114                        Toy Story 2 (1999)
2791                          Airplane! (1980)
2321                      Pleasantville (1998)
1029                              Dumbo (1941)
1197                Princess Bride, The (1987)
594     Snow White and the Seven Dwarfs (1937)
2398             Miracle on 34th Street (1947)
1545                            Ponette (1996)
527                    Schindler's List (1993)
595                Beauty and the Beast (1991)
Name: title, dtype: object

In [16]:
reco = als.recommend(users=[user], dataset=dataset, k=10, filter_viewed=True)
reco

,user_id,item_id,score,rank
0,1,1,0.493485,1
1,1,318,0.464333,2
2,1,110,0.442375,3
3,1,1282,0.383053,4
4,1,1198,0.374360,5
5,1,1196,0.368610,6
6,1,364,0.365436,7
7,1,953,0.340424,8
8,1,1225,0.325510,9
9,1,2087,0.320821,10


In [17]:
titles[reco["item_id"]]

item_id
1                                        Toy Story (1995)
318                      Shawshank Redemption, The (1994)
110                                     Braveheart (1995)
1282                                      Fantasia (1940)
1198                       Raiders of the Lost Ark (1981)
1196    Star Wars: Episode V - The Empire Strikes Back...
364                                 Lion King, The (1994)
953                          It's a Wonderful Life (1946)
1225                                       Amadeus (1984)
2087                                     Peter Pan (1953)
Name: title, dtype: object

- What user 1 really watched next

In [18]:
titles[test[test["user_id"] == user]["item_id"]]

item_id
2687                          Tarzan (1999)
745                   Close Shave, A (1995)
588                          Aladdin (1992)
1                          Toy Story (1995)
2355                   Bug's Life, A (1998)
2294                            Antz (1998)
783     Hunchback of Notre Dame, The (1996)
1566                        Hercules (1997)
1907                           Mulan (1998)
48                        Pocahontas (1995)
Name: title, dtype: object

## More Like This
- Slide 11
- Nearest item vectors to one movie

In [19]:
toy_story = titles.index[titles == "Toy Story (1995)"][0]
similar = als.recommend_to_items(target_items=[toy_story], dataset=dataset, k=10)
titles[similar["item_id"]]

item_id
3114             Toy Story 2 (1999)
34                      Babe (1995)
588                  Aladdin (1992)
2355           Bug's Life, A (1998)
1265           Groundhog Day (1993)
595     Beauty and the Beast (1991)
364           Lion King, The (1994)
2321           Pleasantville (1998)
2687                  Tarzan (1999)
2761         Iron Giant, The (1999)
Name: title, dtype: object

## EASE
- Slide 12
- A linear autoencoder over items: predict each movie from all the others
- Closed form: one matrix inverse, one setting `regularization`

In [20]:
grade("EASE", EASEModel(regularization=500))

{'recall@10': 0.07667218543046359,
 'ndcg@10': 0.08005858814513667,
 'seconds': 3.2}

## SASRec: Predicting the Next Movie
- Slides 13-15
- Each user's history as a sequence, oldest first, the last 200 movies
- Two transformer blocks with two attention heads, and 64 numbers per movie
- Trained to predict the next movie at every position; runs on the GPU when there is one

In [21]:
sasrec = SASRecModel(n_blocks=2, n_heads=2, n_factors=64, session_max_len=200, epochs=40, deterministic=True)
grade("SASRec", sasrec)

{'recall@10': 0.16942052980132452,
 'ndcg@10': 0.18476977323890026,
 'seconds': 42.1}

## Comparing the Recommenders
- Slides 16-17

In [22]:
pd.DataFrame(results).T

,recall@10,ndcg@10,seconds
Popularity,0.041043,0.044436,0.2
Item similarity,0.066987,0.070795,0.5
ALS,0.073179,0.075633,0.6
EASE,0.076672,0.080059,3.2
SASRec,0.169421,0.184770,42.1


## Lab: Your Own Watch History
- Pick a few movies you like, oldest first, and ask SASRec what comes next
- A new user 0 with one row per movie, dated one day apart

In [23]:
mine = ["Toy Story (1995)", "Aladdin (1992)", "Lion King, The (1994)", "Mulan (1998)"]
me = pd.DataFrame({"user_id": 0, "item_id": [titles.index[titles == m][0] for m in mine], "weight": 1.0,
                   "datetime": pd.date_range("2000-01-01", periods=len(mine))})
me

,user_id,item_id,weight,datetime
0,0,1,1.0,2000-01-01
1,0,588,1.0,2000-01-02
2,0,364,1.0,2000-01-03
3,0,1907,1.0,2000-01-04


In [24]:
reco = sasrec.recommend(users=[0], dataset=Dataset.construct(me), k=10, filter_viewed=True)
titles[reco["item_id"]]

item_id
2687                                  Tarzan (1999)
2355                           Bug's Life, A (1998)
2294                                    Antz (1998)
1566                                Hercules (1997)
3114                             Toy Story 2 (1999)
595                     Beauty and the Beast (1991)
2761                         Iron Giant, The (1999)
783             Hunchback of Notre Dame, The (1996)
1405         Beavis and Butt-head Do America (1996)
2700    South Park: Bigger, Longer and Uncut (1999)
Name: title, dtype: object

- Swap in a different taste, such as horror or westerns

## Wrap-Up
- Slides 18-20